In [7]:
# ============================================
# LIMPIEZA UNIFICADA GRID (i-DE, Endesa, Viesgo)
# ============================================

import pandas as pd
from pyproj import Transformer

# -----------------------------
# PATHS (ajustar)
# -----------------------------
PATH_IDE = r"..\..\Raw\Grid_Stations\2026_04_01_R1-001_Demanda.csv"
PATH_ENDESA = r"..\..\Raw\Grid_Stations\2026_04_01_R1005_demanda.csv"
PATH_VIESGO = r"..\..\Raw\Grid_Stations\2026_04_01_R1299_demanda.csv"

# -----------------------------
# HELPERS
# -----------------------------
def clean_column_names(df):
    df = df.copy()
    df.columns = (
        pd.Index(df.columns)
        .astype(str)
        .str.strip()
        .str.replace("ï»¿", "", regex=False)
        .str.replace("Ã³", "ó", regex=False)
        .str.replace("Ã¡", "á", regex=False)
        .str.replace("Ã©", "é", regex=False)
        .str.replace("Ã­", "í", regex=False)
        .str.replace("Ãº", "ú", regex=False)
        .str.replace("Ã±", "ñ", regex=False)
    )
    return df

def clean_numeric_es(series):
    return pd.to_numeric(
        series.astype(str)
        .str.strip()
        .str.replace(".", "", regex=False)
        .str.replace(",", ".", regex=False),
        errors="coerce"
    )

def pick_first_existing_column(df, candidates):
    for col in candidates:
        if col in df.columns:
            return col
    return None

# -----------------------------
# FUNCIÓN GENERAL (UTM -> LAT/LON)
# -----------------------------
def load_distributor_utm(path, distributor_name):
    df = pd.read_csv(path, sep=";", encoding="latin1")
    df = clean_column_names(df)

    # eliminar columnas duplicadas del tipo Provincia.1, Municipio.1, etc.
    df = df.loc[:, ~df.columns.str.endswith(".1")].copy()

    # -------------------------
    # localizar columnas reales
    # -------------------------
    utm_x_col = pick_first_existing_column(df, ["Coordenada UTM X", "UTM X", "X"])
    utm_y_col = pick_first_existing_column(df, ["Coordenada UTM Y", "UTM Y", "Y"])
    cap_col = pick_first_existing_column(df, [
        "Capacidad firme disponible (MW)",
        "Capacidad disponible (MW)",
        "Capacidad de acceso (MW)"
    ])
    node_source_col = pick_first_existing_column(df, [
        "Nombre subestación",
        "Nombre Subestación",
        "Subestación",
        "Subestacion",
        "Nudo",
        "Nodo"
    ])

    missing = []
    if utm_x_col is None:
        missing.append("Coordenada UTM X")
    if utm_y_col is None:
        missing.append("Coordenada UTM Y")
    if cap_col is None:
        missing.append("Capacidad firme disponible (MW)")
    if missing:
        raise KeyError(f"Faltan columnas obligatorias en {distributor_name}: {missing}")

    # -------------------------
    # construir columnas estándar
    # -------------------------
    df["utm_x"] = clean_numeric_es(df[utm_x_col])
    df["utm_y"] = clean_numeric_es(df[utm_y_col])
    df["capacity_mw"] = clean_numeric_es(df[cap_col])

    if node_source_col is not None:
        df["node_id"] = df[node_source_col].astype(str).str.strip()
        df.loc[df["node_id"].isin(["", "nan", "None"]), "node_id"] = pd.NA
    else:
        df["node_id"] = pd.NA

    # fallback si falta node_id
    missing_node_mask = df["node_id"].isna()
    if missing_node_mask.any():
        df.loc[missing_node_mask, "node_id"] = [
            f"{distributor_name}_{i:04d}" for i in range(missing_node_mask.sum())
        ]

    # -------------------------
    # filtrar nulos / inválidos
    # -------------------------
    df = df.dropna(subset=["utm_x", "utm_y", "capacity_mw"]).copy()
    df = df[df["capacity_mw"] > 0].copy()

    # -------------------------
    # convertir UTM -> WGS84
    # -------------------------
    transformer = Transformer.from_crs("EPSG:25830", "EPSG:4326", always_xy=True)
    lon, lat = transformer.transform(df["utm_x"].values, df["utm_y"].values)

    df["longitude"] = lon
    df["latitude"] = lat

    # -------------------------
    # quedarnos solo con columnas finales antes de agrupar
    # -------------------------
    df = df[["node_id", "latitude", "longitude", "capacity_mw"]].copy()

    # defensa extra por si hubiera nombres duplicados
    df = df.loc[:, ~df.columns.duplicated()].copy()

    # -------------------------
    # agregar por nodo
    # -------------------------
    df = (
        df.groupby(["node_id", "latitude", "longitude"], as_index=False)
          .agg({"capacity_mw": "sum"})
    )

    # -------------------------
    # salida final
    # -------------------------
    df["distributor"] = distributor_name
    df["capacity_kw"] = df["capacity_mw"] * 1000

    df = df[[
        "distributor",
        "node_id",
        "latitude",
        "longitude",
        "capacity_mw",
        "capacity_kw"
    ]].reset_index(drop=True)

    return df

# -----------------------------
# CARGA DE LOS 3 DISTRIBUIDORES
# -----------------------------
df_ide = load_distributor_utm(PATH_IDE, "i-DE")
df_endesa = load_distributor_utm(PATH_ENDESA, "Endesa")
df_viesgo = load_distributor_utm(PATH_VIESGO, "Viesgo")

# -----------------------------
# UNIFICACIÓN FINAL
# -----------------------------
df_grid_clean = pd.concat(
    [df_ide, df_endesa, df_viesgo],
    ignore_index=True
).drop_duplicates()

# -----------------------------
# CHECKS
# -----------------------------
print("Distributors:")
print(df_grid_clean["distributor"].value_counts())

print("\nShape total:", df_grid_clean.shape)

print("\nNulls:")
print(df_grid_clean.isna().sum())

print("\nCapacity MW:")
print(df_grid_clean["capacity_mw"].describe())

# -----------------------------
# EXPORT
# -----------------------------
OUTPUT_PATH = r"..\..\Processed\grid_cleaned\grid_clean.parquet"
df_grid_clean.to_parquet(OUTPUT_PATH, index=False)

print(f"\nSaved in: {OUTPUT_PATH}")

Distributors:
distributor
i-DE      112
Endesa     72
Viesgo     65
Name: count, dtype: int64

Shape total: (249, 6)

Nulls:
distributor    0
node_id        0
latitude       0
longitude      0
capacity_mw    0
capacity_kw    0
dtype: int64

Capacity MW:
count    249.000000
mean      20.006787
std       23.609193
min        0.060000
25%        3.010000
50%       11.400000
75%       27.470000
max      129.270000
Name: capacity_mw, dtype: float64

Saved in: ..\..\Processed\grid_cleaned\grid_clean.parquet
